# 16 - Enterprise Data Governance & Unity Catalog
### Masterclass: 3-Level Namespace, Fine-Grained Access Control (RLS & CLS), Lineage, Audit System Tables, and Delta Sharing

---

## Why Governance is a Top-Tier Senior Data Engineer Interview Topic

In modern enterprise architectures (especially Databricks Lakehouse, Snowflake, and BigQuery), **data pipelines cannot exist without data governance**. A Senior Data Engineer is expected to know far more than just writing Spark jobs; you must understand how data is **secured, audited, classified, and shared** across the organization without creating operational bottlenecks.

```text
 ┌────────────────────────────────────────────────────────────────────────┐
 │                   ENTERPRISE DATA GOVERNANCE PILLARS                   │
 ├────────────────────────────────────────────────────────────────────────┤
 │ 1. Unified Metadata:   Centralized Metastore across all workspaces/clouds│
 │ 2. Access Control:     Role-Based (RBAC) + Attribute-Based (ABAC)      │
 │ 3. Fine-Grained Sec:   Row Filters (RLS) + Dynamic Column Masks (CLS)  │
 │ 4. Privacy Compliance: HIPAA (PHI), GDPR (Right to Forget), SOC 2      │
 │ 5. Data Lineage:       Automatic runtime table & column dependency DAGs│
 │ 6. Auditing & Cost:    System Tables (system.access.audit, billing)    │
 │ 7. Open Sharing:       Zero-copy external data exchange (Delta Sharing)│
 │ 8. Unstructured Data:  Governed Volumes for PDFs, images, and models   │
 └────────────────────────────────────────────────────────────────────────┘
```

---

### The Unity Catalog Object Hierarchy

```mermaid
graph TD
    Account[Databricks Account Level] --> Metastore[Unity Catalog Metastore: Shared across workspaces & regions]
    Metastore --> StorageCred[Storage Credentials: IAM Roles / Service Principals]
    Metastore --> ExtLoc[External Locations: S3 Bucket / Azure ADLS Paths]
    Metastore --> CatProd[Catalog: prod_lakehouse]
    Metastore --> CatDev[Catalog: dev_lakehouse]
    
    CatProd --> SchemaBronze[Schema: bronze]
    CatProd --> SchemaSilver[Schema: silver]
    CatProd --> SchemaGold[Schema: gold]
    
    SchemaSilver --> TblClaims[Table: claims]
    SchemaSilver --> VolImages[Volume: medical_scans_raw]
    SchemaSilver --> FuncMask[Function: mask_ssn_fn]
    
    TblClaims --> RLS[Row Filter: provider_isolation_fn]
    TblClaims --> CLS[Column Mask: mask_phi_ssn]
```

---

### What You Will Learn & Practice in This Notebook:
1. **The Unity Catalog Architecture:** 3-Level namespace (`catalog.schema.table`), Account Metastore, and Storage Credentials.
2. **Access Control & Privilege Inheritance:** `GRANT`, `REVOKE`, Role-Based Access Control, and the Principle of Least Privilege.
3. **Fine-Grained Access Control (FGAC):** Writing and attaching **Row Filters** (Row-Level Security) and **Dynamic Column Masks** (Column-Level Security).
4. **Data Tagging & Regulatory Compliance:** HIPAA Minimum Necessary rule, PHI masking, and GDPR Delta Lake hard-delete tombstoning.
5. **Automated Lineage Tracking:** Extracting table-level and column-level lineage from execution graphs.
6. **Audit Logging & System Tables:** Querying `system.access.audit` and `system.billing.usage` to detect unauthorized access anomalies.
7. **Delta Sharing & Clean Rooms:** Secure, zero-copy sharing across cloud providers without replication.
8. **Governing Unstructured Data with Volumes:** Managed vs External Volumes for medical imaging and reports.
9. **Senior DE Interview Playbook:** Real-world governance interview questions, common traps, and the 2-minute executive pitch.


In [ ]:
import sys, pathlib, os, json, time, datetime, hashlib
import pandas as pd
from pyspark.sql import functions as F, types as T, Window

# Locate repo root and register src/ directory
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

from de_helpers import (
    q, q1, assert_grain, tables, columns, explain, display,
    get_spark, get_dbutils, plan_string, has_plan_operator
)

spark = get_spark("16-data-governance")
dbutils = get_dbutils()

print("Setup complete. Practice environment ready.")
display(tables())


---
## Module 1: The Unity Catalog Architecture & 3-Level Namespace

### Legacy Hive Metastore vs. Modern Unity Catalog

| Dimension | Legacy Hive Metastore (HMS) | Modern Unity Catalog (UC) |
| :--- | :--- | :--- |
| **Namespace** | 2-level (`schema.table` or `database.table`) | **3-level** (`catalog.schema.table` or `catalog.schema.volume`) |
| **Scope** | Bound to a single Databricks workspace | **Account-level**: One central catalog shared across all workspaces and clouds |
| **Identity** | Workspace-local users | Account-level unified users, groups, and Service Principals (via SCIM/SSO) |
| **Storage Access** | Direct IAM instance profiles / cluster keys | **Storage Credentials & External Locations** managed via SQL grants |
| **Unstructured Data** | Unmanaged DBFS or raw S3 mounts | **Volumes** (Managed & External) with full SQL access controls and auditing |
| **Security Granularity**| Table or View level only | Row Filters, Column Masks, Column-level tags, and Table constraints |
| **Lineage** | Requires manual third-party agents | **Native, automatic runtime capture** down to column-level |

---

### The 3-Level Namespace Hierarchy
Every securable asset in the Lakehouse is uniquely referenced using three identifiers:

$$\text{catalog} \;\; \boldsymbol{.} \;\; \text{schema} \;\; \boldsymbol{.} \;\; \text{securable\_object}$$

1. **Catalog (`prod_lakehouse`):** The primary isolation boundary (e.g. environment: `prod`, `dev`, `analytics`, or business unit: `claims`, `pharmacy`).
2. **Schema / Database (`silver`):** Logical grouping of tables, views, volumes, and functions (e.g. Medallion layer or domain).
3. **Securable Object (`claims`):** The actual table, view, managed volume, or user-defined function.

```sql
-- Querying across catalogs and schemas seamlessly:
SELECT c.claim_id, p.provider_name
FROM prod_healthcare.silver.claims c
INNER JOIN prod_healthcare.gold.dim_providers p
  ON c.provider_id = p.provider_id;
```


In [ ]:
# Inspecting 3-Level Namespace & Catalog Simulation

# In local practice environments (DuckDB / local Spark), the active catalog is 'workspace' or 'main'.
# Let's inspect the active tables and their fully qualified 3-level names:

catalog_metadata = q("""
    SELECT 
        'healthcare_catalog' AS catalog_name,
        table_schema AS schema_name,
        table_name,
        table_type,
        'healthcare_catalog.' || table_schema || '.' || table_name AS fully_qualified_3_level_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""")

print("=== Fully Qualified 3-Level Namespace Assets in Catalog ===")
display(catalog_metadata)

# Demonstrate 3-level referencing in PySpark
try:
    spark_tables = spark.sql("SHOW TABLES IN workspace.healthcare").toPandas()
    print("\nDatabricks Unity Catalog Tables:")
    display(spark_tables)
except Exception:
    print("\nLocal Spark session active: Unity Catalog namespace mapped successfully.")


---
## Module 2: Role-Based Access Control (RBAC) & Privilege Inheritance

### The Principle of Least Privilege in Data Engineering
In an enterprise, users should never connect with generic admin or root privileges. Access must be granted to **groups** or **service principals**, not individual user accounts.

```mermaid
graph TD
    Metastore[Metastore] -->|Inherits| Catalog[Catalog: prod_claims]
    Catalog -->|Inherits| Schema[Schema: silver]
    Schema -->|Inherits| Table[Table: claims]
    
    Admin[Group: data_engineers_prod] -.->|USE CATALOG, USE SCHEMA, MODIFY| Schema
    Analyst[Group: financial_analysts] -.->|USE CATALOG, USE SCHEMA, SELECT| Table
```

### Privilege Inheritance Rules in Unity Catalog
1. **`USE CATALOG` is Mandatory:** Even if a user has `SELECT` on a table, they **cannot read it** unless they also possess `USE CATALOG` on the parent catalog AND `USE SCHEMA` on the parent schema!
2. **Inheritance Flows Down:** Granting `SELECT` on a Schema grants read access to **all existing and future tables** within that schema.
3. **No Negative Privileges:** There is no explicit `DENY` in Unity Catalog. Access is granted; lack of grant is denial.

### Core Unity Catalog Privileges Reference:
* `USE CATALOG`: Allows traversing into the catalog.
* `USE SCHEMA`: Allows traversing into the schema.
* `SELECT`: Allows reading data from a table, view, or stream.
* `MODIFY`: Allows `INSERT`, `UPDATE`, `DELETE`, and `MERGE INTO`.
* `CREATE TABLE` / `CREATE VOLUME`: Allows creating new assets in a schema.
* `EXECUTE`: Allows executing a user-defined function (UDF) or stored procedure.
* `APPLY TAG`: Allows tagging assets with metadata tags.
* `ALL PRIVILEGES`: Grants all permissions on the target object (bestowed upon object owner).


In [ ]:
# Simulating the Unity Catalog Privilege Evaluation Engine

class UnityCatalogRBACSimulator:
    """Simulates Unity Catalog hierarchical permission evaluation"""
    def __init__(self):
        # Grants dictionary: (principal, asset_path) -> set of privileges
        self.grants = {}

    def grant(self, principal, privilege, asset_path):
        key = (principal, asset_path)
        if key not in self.grants:
            self.grants[key] = set()
        self.grants[key].add(privilege)
        print(f"[GRANT] {privilege} ON {asset_path} TO {principal}")

    def check_access(self, principal, action, fq_table_name):
        """
        To execute 'SELECT' on catalog.schema.table, the principal must have:
        1. USE CATALOG on catalog
        2. USE SCHEMA on catalog.schema
        3. SELECT on catalog.schema.table OR on catalog.schema OR on catalog
        """
        parts = fq_table_name.split(".")
        catalog, schema, table = parts[0], f"{parts[0]}.{parts[1]}", fq_table_name
        
        # Check hierarchy
        has_use_cat = "USE CATALOG" in self.grants.get((principal, catalog), set()) or "ALL PRIVILEGES" in self.grants.get((principal, catalog), set())
        has_use_sch = "USE SCHEMA" in self.grants.get((principal, schema), set()) or "ALL PRIVILEGES" in self.grants.get((principal, schema), set())
        
        has_obj_priv = (
            action in self.grants.get((principal, table), set()) or
            action in self.grants.get((principal, schema), set()) or
            action in self.grants.get((principal, catalog), set()) or
            "ALL PRIVILEGES" in self.grants.get((principal, table), set())
        )
        
        is_authorized = has_use_cat and has_use_sch and has_obj_priv
        print(f"Principal '{principal}' attempting {action} on {fq_table_name}: ")
        print(f"  - USE CATALOG: {has_use_cat} | USE SCHEMA: {has_use_sch} | {action} PRIVILEGE: {has_obj_priv}")
        print(f"  ==> Result: {'AUTHORIZED (Access Granted)' if is_authorized else 'DENIED (Insufficient Privileges)'}\n")
        return is_authorized

rbac = UnityCatalogRBACSimulator()

print("=== Configuring Enterprise Lakehouse Privileges ===")
# Grant Analyst: only table SELECT without USE CATALOG/SCHEMA (A COMMON BUG!)
rbac.grant("bi_analyst_group", "SELECT", "prod.silver.claims")
# Check access before granting path traversal
rbac.check_access("bi_analyst_group", "SELECT", "prod.silver.claims")

# Now correctly grant traversal privileges
rbac.grant("bi_analyst_group", "USE CATALOG", "prod")
rbac.grant("bi_analyst_group", "USE SCHEMA", "prod.silver")
# Re-evaluate
rbac.check_access("bi_analyst_group", "SELECT", "prod.silver.claims")


---
## Module 3: Fine-Grained Access Control (Row Filters & Column Masks)

### Beyond Binary Table Permissions
Traditionally, if an analyst had `SELECT` on `claims`, they could view every row and every column. To protect Protected Health Information (PHI) and isolate tenants, companies created dozens of brittle, expensive views:
* `vw_claims_florida`
* `vw_claims_texas`
* `vw_claims_masked_ssn`

> [!IMPORTANT]
> **Unity Catalog eliminates View Proliferation** through **Row Filters** (Row-Level Security / RLS) and **Dynamic Column Masks** (Column-Level Security / CLS) applied directly to the underlying Delta table!

---

### 1. Row Filters (Row-Level Security - RLS)
A Row Filter is a user-defined SQL scalar function that accepts one or more column values and returns a `BOOLEAN`. Unity Catalog transparently appends this condition to the query's `WHERE` clause at runtime!

```sql
-- 1. Define the security filter function
CREATE OR REPLACE FUNCTION prod.security.provider_isolation_filter(provider_id_val STRING)
RETURN 
  IS_ACCOUNT_GROUP_MEMBER('claims_admin') OR 
  CURRENT_USER() = provider_id_val || '@healthcare-portal.com';

-- 2. Attach the filter to the target table
ALTER TABLE prod.silver.claims 
SET ROW FILTER prod.security.provider_isolation_filter ON (provider_id);
```
* **Effect:** When a doctor logs in, they only see claims matching their own `provider_id`. An administrator sees all claims. Zero separate views needed!

---

### 2. Dynamic Column Masks (Column-Level Security - CLS)
A Column Mask is a user-defined scalar function that accepts a column value and returns the transformed/masked value depending on who is querying it:

```sql
-- 1. Define the masking function for Patient Social Security Number
CREATE OR REPLACE FUNCTION prod.security.mask_ssn_fn(ssn_val STRING)
RETURN 
  CASE 
    WHEN IS_ACCOUNT_GROUP_MEMBER('compliance_officers') THEN ssn_val
    WHEN IS_ACCOUNT_GROUP_MEMBER('claims_adjudicators') THEN '***-**-' || RIGHT(ssn_val, 4)
    ELSE '***-**-****'
  END;

-- 2. Bind the mask to the column
ALTER TABLE prod.silver.members 
ALTER COLUMN ssn SET MASK prod.security.mask_ssn_fn;
```
* **Effect:** Compliance auditors see plaintext `123-45-6789`; analysts see `***-**-6789`; unauthorized BI users see `***-**-****`. The physical parquet files remain untouched; masking is applied dynamically in-memory during Catalyst plan generation!


In [ ]:
# Live Demonstration of Fine-Grained Security: Row Filters & Column Masking

# 1. Prepare Base Member Table with Sensitive PHI (Member ID, DOB, SSN, Zip, State)
members_df = q("""
    SELECT 
        member_id,
        plan_id,
        enrollment_date,
        dob AS date_of_birth,
        '987-65-4321' AS ssn,
        '33101' AS zip_code,
        state
    FROM members
    LIMIT 6
""")

# 2. Implement Dynamic Column Mask Function
def apply_column_mask(df, user_role):
    """Applies dynamic masking based on caller's role"""
    result = df.copy()
    if user_role == "compliance_officer":
        # Plaintext access granted for regulatory audits
        pass
    elif user_role == "claims_analyst":
        # Partial mask: Show only last 4 of SSN and Year of Birth (HIPAA safe-harbor)
        result["ssn"] = result["ssn"].apply(lambda s: f"***-**-{s[-4:]}")
        result["date_of_birth"] = result["date_of_birth"].apply(lambda d: f"{str(d)[:4]}-**-**")
        result["zip_code"] = result["zip_code"].apply(lambda z: f"{z[:3]}**")
    else:
        # Full redaction for standard BI consumers
        result["ssn"] = "***-**-****"
        result["date_of_birth"] = "****-**-**"
        result["zip_code"] = "*****"
    return result

# 3. Implement Row Filter Function (State / Jurisdiction Isolation)
def apply_row_filter(df, user_role, user_state_scope=None):
    """Applies tenant row filter: Florida team only sees FL members"""
    if user_role == "super_admin":
        return df
    if user_state_scope:
        return df[df["state"] == user_state_scope].copy()
    return df

print("=== 1. Compliance Officer View (Full Plaintext Access) ===")
display(apply_column_mask(members_df, "compliance_officer"))

print("\n=== 2. Claims Analyst View (Dynamic HIPAA Mask Applied) ===")
display(apply_column_mask(members_df, "claims_analyst"))

print("\n=== 3. Florida Regional Auditor View (Row Filter + Mask Applied) ===")
fl_filtered = apply_row_filter(members_df, user_role="regional_auditor", user_state_scope="FL")
display(apply_column_mask(fl_filtered, "standard_bi_user"))


---
## Module 4: Data Classification, Tagging & Privacy Regulations (HIPAA & GDPR)

### Data Classification Standards in Healthcare Lakehouses
Unity Catalog supports **System Tags** and **Custom Tags** at the catalog, schema, table, and column levels.

| Tag Key | Example Values | Purpose & Enforcement |
| :--- | :--- | :--- |
| `data_classification` | `PUBLIC`, `INTERNAL`, `CONFIDENTIAL`, `RESTRICTED` | Determines exportability and zero-copy sharing rights. |
| `phi` | `TRUE`, `FALSE` | Triggers HIPAA audit logging and mandatory encryption at rest. |
| `pii_type` | `SSN`, `DOB`, `EMAIL`, `ADDRESS`, `FINANCIAL` | Automatically binds dynamic masking functions to the column. |
| `data_domain` | `CLAIMS`, `CLINICAL`, `ENROLLMENT`, `PHARMACY` | Data mesh domain ownership and data product attribution. |

---

### The GDPR "Right to be Forgotten" in an Append-Only Lakehouse

> [!WARNING]
> **The Delta Lake Storage Dilemma:**
> Delta Lake is built on immutable Parquet files. When you run `DELETE FROM silver.members WHERE member_id = 'MBR001'`, Delta does **NOT** immediately delete the parquet file—it writes a new parquet file without the row and updates the Delta Transaction Log!
> Historical versions in the transaction log still contain the deleted person's data for Time Travel!

#### The Compliant 3-Step "Right to be Forgotten" Protocol:
1. **Targeted Hard Delete:**
   ```sql
   DELETE FROM silver.members WHERE member_id = 'MBR999';
   DELETE FROM silver.claims  WHERE member_id = 'MBR999';
   ```
2. **Purge Historical Commits (Tombstoning & Retention Tuning):**
   ```sql
   -- Reduce retention period specifically for GDPR compliance:
   ALTER TABLE silver.members SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');
   ```
3. **Vacuum Physical Storage:**
   ```sql
   VACUUM silver.members RETAIN 0 HOURS;
   VACUUM silver.claims RETAIN 0 HOURS;
   ```
   * Result: All older parquet files containing the member's data are permanently wiped from S3/ADLS.


In [ ]:
# Simulating Data Classification Tagging & GDPR Deletion Protocol

# 1. Column Tagging Registry
data_dictionary_tags = pd.DataFrame([
    {"table": "members", "column": "member_id", "tag_key": "pii_type", "tag_value": "PSEUDONYM_ID", "phi": False},
    {"table": "members", "column": "ssn", "tag_key": "pii_type", "tag_value": "DIRECT_IDENTIFIER", "phi": True},
    {"table": "members", "column": "date_of_birth", "tag_key": "pii_type", "tag_value": "QUASI_IDENTIFIER", "phi": True},
    {"table": "claims", "column": "claim_id", "tag_key": "data_domain", "tag_value": "FINANCIAL_TRANSACTION", "phi": False},
    {"table": "claims", "column": "primary_diagnosis_code", "tag_key": "clinical_type", "tag_value": "ICD10_CM", "phi": True},
    {"table": "claims", "column": "paid_amount", "tag_key": "confidentiality", "tag_value": "HIGH_COMMERCIAL", "phi": False},
])

print("=== Unity Catalog Data Classification Catalog ===")
display(data_dictionary_tags)

# 2. GDPR Hard-Delete Simulation
print("\n=== Simulating GDPR 'Right to be Forgotten' Erasure Request ===")
erasure_target_member = "MBR00615"

# Verify member presence before deletion
before_count = q1(f"SELECT COUNT(*) FROM claims WHERE member_id = '{erasure_target_member}'")
print(f"Target Member '{erasure_target_member}' Claims Found in Silver: {before_count} claims")

# Generate compliant audit record for Data Protection Officer (DPO)
gdpr_erasure_audit = {
    "request_id": "GDPR-2026-10-11-8842",
    "subject_id": erasure_target_member,
    "request_timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "tables_affected": ["silver.members", "silver.claims", "gold.fact_member_utilization"],
    "records_erased": {"silver.claims": before_count, "silver.members": 1},
    "storage_action": "DELETE + DELTA VACUUM RETAIN 0 HOURS",
    "status": "VERIFIED_PURGED_FROM_LAKEHOUSE"
}

print("\n--- DPO Compliance Audit Certificate ---")
print(json.dumps(gdpr_erasure_audit, indent=2))


---
## Module 5: Automated End-to-End Data Lineage & Impact Analysis

### How Unity Catalog Captures Lineage Without User Instrumentation
Unlike legacy data catalogs (e.g. Apache Atlas, Collibra) that required engineers to manually tag inputs and outputs or emit OpenLineage events, **Unity Catalog captures lineage automatically at the Spark Catalyst execution level**.

When Spark analyzes and compiles a logical query plan, Unity Catalog inspects the `LogicalRelation` nodes (sources) and the target `InsertIntoDeltaTable` / `CreateViewCommand` nodes.

```text
 ┌─────────────────┐       ┌─────────────────┐       ┌────────────────────────┐
 │  raw.claims_s3  │ ───>  │  silver.claims  │ ───>  │gold.fact_daily_provider│
 └─────────────────┘       └─────────────────┘       └────────────────────────┘
          ▲                         ▲                             ▲
          │                         │                             │
    [File Lineage]            [Table Lineage]             [Column Lineage]
   (S3 raw paths)          (Auto Loader merge)       (c.paid_amount -> sum_paid)
```

### Table Lineage vs. Column Lineage
1. **Table-Level Lineage:** Identifies which tables contributed to an output table. Used for **Break-Glass Root Cause Analysis** (e.g. "Upstream table `silver.claims` had bad data; which 15 downstream Gold marts are infected?").
2. **Column-Level Lineage:** Identifies the precise expression lineage of each column.
   * Example: `gold.fact_daily_provider.total_paid` is derived from `SUM(silver.claims.paid_amount)`.
   * Essential for **GDPR impact tracing** and **Regulatory Auditing**.


In [ ]:
# Simulating Automatic Column & Table Lineage Extraction from Spark Query

claims_sp = spark.table("claims")
providers_sp = spark.table("providers")

# Downstream Gold Transformation Query
gold_mart_df = (
    claims_sp.filter(F.col("status") == "Approved")
    .join(providers_sp, on="provider_id", how="inner")
    .groupBy(
        F.col("claim_date").alias("service_date"),
        F.col("provider_id"),
        F.col("provider_name")
    )
    .agg(
        F.count("claim_id").alias("paid_claim_count"),
        F.round(F.sum("paid_amount"), 2).alias("total_paid_amount")
    )
)

# Extract and display the physical execution plan
plan_str = plan_string(gold_mart_df)

# Construct queryable Lineage Graph DataFrame
lineage_edges = pd.DataFrame([
    {"source_table": "silver.claims", "source_column": "claim_date", "target_table": "gold.fact_daily_provider", "target_column": "service_date", "transformation_type": "DIRECT_PROJECTION"},
    {"source_table": "silver.claims", "source_column": "provider_id", "target_table": "gold.fact_daily_provider", "target_column": "provider_id", "transformation_type": "JOIN_KEY"},
    {"source_table": "silver.providers", "source_column": "provider_name", "target_table": "gold.fact_daily_provider", "target_column": "provider_name", "transformation_type": "ENRICHMENT"},
    {"source_table": "silver.claims", "source_column": "claim_id", "target_table": "gold.fact_daily_provider", "target_column": "paid_claim_count", "transformation_type": "COUNT_AGGREGATION"},
    {"source_table": "silver.claims", "source_column": "paid_amount", "target_table": "gold.fact_daily_provider", "target_column": "total_paid_amount", "transformation_type": "SUM_AGGREGATION"},
])

print("=== Unity Catalog Automated Column Lineage Graph ===")
display(lineage_edges)

# Impact Analysis Query: If 'silver.claims.paid_amount' changes definition, what downstream columns break?
impacted_kpis = lineage_edges[lineage_edges["source_column"] == "paid_amount"]
print("\n--- Upstream Impact Analysis for 'silver.claims.paid_amount' ---")
display(impacted_kpis[["target_table", "target_column", "transformation_type"]])


---
## Module 6: Audit Logging & Databricks System Tables

### What are System Tables?
In Databricks Unity Catalog, all audit logs, billable usage, job runs, and access telemetry are stored natively as **queryable Delta tables** under the `system` catalog!
You no longer need to scrape AWS CloudTrail or configure complex log forwarding to answer compliance questions.

```text
 ┌────────────────────────────────────────────────────────────────────────┐
 │                    SYSTEM CATALOG (system.*)                           │
 ├────────────────────────────────────────────────────────────────────────┤
 │ system.access.audit          - Every SQL query, API call, login, grant │
 │ system.billing.usage         - DBU consumption by job, cluster, user   │
 │ system.lineage.table_lineage - Relational table lineage graph          │
 │ system.lineage.column_lineage- Column-level derivation graph           │
 │ system.lakeflow.jobs         - Workflow and job execution telemetry    │
 └────────────────────────────────────────────────────────────────────────┘
```

---

### The 4 Essential Compliance Queries Data Engineers Run:
1. **Who Queried PHI / Sensitive Tables in the Last 24 Hours?**
   ```sql
   SELECT user_identity.email, action_name, request_params.table_full_name, event_time
   FROM system.access.audit
   WHERE request_params.table_full_name = 'prod.silver.members'
     AND event_time >= CURRENT_TIMESTAMP() - INTERVAL 24 HOURS;
   ```
2. **Detecting Bulk Export Anomalies:**
   * Alert if a user runs a `SELECT *` returning > 500,000 rows on a customer table.
3. **Tracking Privilege Escalations:**
   * Who granted `ALL PRIVILEGES` or altered row filters?
4. **Attributing Lakehouse Compute Cost:**
   * Grouping `system.billing.usage` by `custom_tags.team` to charge compute back to business units.


In [ ]:
# Simulating System Audit Log Analytics & Security Breach Detection

# 1. Synthesize system.access.audit records
audit_logs_sample = pd.DataFrame([
    {"event_time": "2026-10-11 10:14:02", "user_email": "analyst_sarah@healthcare.com", "action_name": "commandSubmit", "table_accessed": "silver.claims", "rows_read": 120, "status": "SUCCESS", "client_ip": "192.168.1.45"},
    {"event_time": "2026-10-11 11:30:19", "user_email": "intern_dave@healthcare.com",   "action_name": "commandSubmit", "table_accessed": "silver.members", "rows_read": 50, "status": "SUCCESS", "client_ip": "192.168.1.88"},
    {"event_time": "2026-10-11 14:22:45", "user_email": "service_principal_etl@azure", "action_name": "mergeDelta",     "table_accessed": "silver.claims", "rows_read": 950000, "status": "SUCCESS", "client_ip": "10.0.4.12"},
    {"event_time": "2026-10-11 18:45:10", "user_email": "contractor_bob@external.com", "action_name": "commandSubmit", "table_accessed": "silver.members", "rows_read": 850000, "status": "SUCCESS", "client_ip": "198.51.100.22"}, # ANOMALY: Mass download of PII
    {"event_time": "2026-10-11 19:05:00", "user_email": "unknown_user@tor_proxy.net",   "action_name": "commandSubmit", "table_accessed": "silver.members", "rows_read": 0, "status": "DENIED_PERMISSION", "client_ip": "203.0.113.195"}, # Denied breach attempt
])

# 2. Automated SOC 2 Anomaly Detection Query
print("=== SOC 2 Compliance Audit: High-Volume Sensitive Data Extraction ===")
sensitive_bulk_reads = audit_logs_sample[
    (audit_logs_sample["table_accessed"] == "silver.members") &
    (audit_logs_sample["rows_read"] > 100000)
]
display(sensitive_bulk_reads)

# 3. Privilege Abuse / Unauthorized Access Attempts
print("\n=== Security Audit: Denied Access Attempts ===")
denied_attempts = audit_logs_sample[audit_logs_sample["status"] == "DENIED_PERMISSION"]
display(denied_attempts)

assert len(sensitive_bulk_reads) == 1, "Anomaly detection failed to catch bulk contractor download!"
print("\nSECURITY ALERT TRIGGERED: Contractor attempted bulk extraction of 850,000 member records!")


---
## Module 7: Delta Sharing & Secure Collaboration (Zero-Copy Open Sharing)

### The Legacy Problem of Data Sharing
Historically, when a health insurer wanted to share adjudicated claim summaries with corporate employers or actuarial partners:
* They generated CSV files and pushed them via **SFTP**.
* They maintained fragile **batch REST APIs**.
* They duplicated petabytes into separate AWS S3 or Azure ADLS accounts.

> [!IMPORTANT]
> **What is Delta Sharing?**
> Delta Sharing is the world's first **open protocol for secure, zero-copy data sharing**.
> It allows sharing live Delta tables directly with external partners **without copying data**, across different cloud providers (AWS $\to$ Azure $\to$ GCP), and to non-Databricks platforms (Power BI, pandas, Apache Spark, Snowflake).

---

### The Delta Sharing Architecture

```text
 ┌────────────────────────────────────────────────────────┐
 │            DATABRICKS UNITY CATALOG PROVIDER           │
 │               Catalog: prod_healthcare                │
 └───────────────────────────┬────────────────────────────┘
                             │  1. Admin defines SHARE:
                             │     CREATE SHARE employer_share;
                             │     ALTER SHARE employer_share ADD TABLE gold.employer_summary;
                             ▼
 ┌────────────────────────────────────────────────────────┐
 │                    DELTA SHARE ASSET                   │
 │           Restricted to defined recipient token        │
 └───────────────────────────┬────────────────────────────┘
                             │  2. Direct Pre-signed S3 URL generated
                             │     (Zero copy, TLS 1.3 encrypted, auditable)
                             ▼
         ┌───────────────────┼───────────────────┐
         ▼                   ▼                   ▼
 ┌───────────────┐   ┌───────────────┐   ┌───────────────┐
 │ Power BI User │   │ Python Pandas │   │ Snowflake DW  │
 │  (Corporate)  │   │  (Actuaries)  │   │   (Partner)   │
 └───────────────┘   └───────────────┘   └───────────────┘
```

#### SQL Syntax for Delta Sharing:
```sql
-- 1. Create a Share
CREATE SHARE IF NOT EXISTS claims_actuarial_share;

-- 2. Add specific tables or partitions to the Share
ALTER SHARE claims_actuarial_share 
ADD TABLE prod_healthcare.gold.fact_claims_daily_provider
AS SELECT * FROM prod_healthcare.gold.fact_claims_daily_provider
WHERE service_date >= '2026-01-01';

-- 3. Create a Recipient (generates activation credential link)
CREATE RECIPIENT IF NOT EXISTS milliman_actuaries;

-- 4. Grant access
GRANT SELECT ON SHARE claims_actuarial_share TO RECIPIENT milliman_actuaries;
```


In [ ]:
# Simulating Delta Sharing Provider-Recipient Entitlement Verification

class DeltaSharingSimulator:
    """Simulates open Delta Sharing protocol handshake"""
    def __init__(self):
        self.shares = {}
        self.recipients = {}

    def create_share(self, share_name):
        self.shares[share_name] = {"tables": {}, "recipients": set()}
        print(f"[SHARE CREATED] {share_name}")

    def add_table_to_share(self, share_name, table_name, partition_filter=None):
        self.shares[share_name]["tables"][table_name] = partition_filter
        print(f"[TABLE ADDED] {table_name} (Filter: {partition_filter}) -> Share: {share_name}")

    def create_recipient(self, recipient_name, bearer_token):
        self.recipients[recipient_name] = {"token": bearer_token, "accessible_shares": set()}
        print(f"[RECIPIENT CREATED] {recipient_name} (Bearer Token Registered)")

    def grant_share_to_recipient(self, share_name, recipient_name):
        self.shares[share_name]["recipients"].add(recipient_name)
        self.recipients[recipient_name]["accessible_shares"].add(share_name)
        print(f"[GRANT] SELECT ON SHARE {share_name} TO RECIPIENT {recipient_name}")

    def client_query_share(self, recipient_name, provided_token, share_name, target_table):
        """Simulates an external client (Pandas/Snowflake) requesting data"""
        if recipient_name not in self.recipients or self.recipients[recipient_name]["token"] != provided_token:
            return "401 UNAUTHORIZED: Invalid or expired bearer credential."
        if share_name not in self.recipients[recipient_name]["accessible_shares"]:
            return "403 FORBIDDEN: Recipient has no entitlement on this Share."
        if target_table not in self.shares[share_name]["tables"]:
            return "404 NOT FOUND: Table is not exposed in this Share."
        
        filter_spec = self.shares[share_name]["tables"][target_table]
        return f"200 OK: Pre-signed parquet URLs returned for {target_table} (Applied Partition: {filter_spec})"

ds = DeltaSharingSimulator()

print("=== Configuring Delta Sharing ===")
ds.create_share("corporate_wellness_share")
ds.add_table_to_share("corporate_wellness_share", "gold.fact_claims_daily_provider", "service_date >= '2026-01-01'")
ds.create_recipient("acme_corp_hr", bearer_token="secret_tok_acme_8841")
ds.grant_share_to_recipient("corporate_wellness_share", "acme_corp_hr")

print("\n=== Testing External Access Handshakes ===")
# 1. Authorized external call
res_auth = ds.client_query_share("acme_corp_hr", "secret_tok_acme_8841", "corporate_wellness_share", "gold.fact_claims_daily_provider")
print(f"Authorized Client Response: {res_auth}")

# 2. Tampered Token Call
res_tampered = ds.client_query_share("acme_corp_hr", "wrong_token_999", "corporate_wellness_share", "gold.fact_claims_daily_provider")
print(f"Tampered Client Response:   {res_tampered}")


---
## Module 8: Governing Unstructured Data with Volumes

### The Unstructured Data Blindspot
Data platforms don't just store tabular parquet files; enterprise health systems process:
* Medical imaging (DICOM / MRI scans)
* Physician clinical notes (PDF / Word documents)
* Audio recordings of telehealth consultations
* Pre-trained ML / LLM model weights

In legacy environments, engineers used **DBFS mounts** (`/mnt/s3_bucket/`). DBFS mounts had severe security risks: all users on a high-concurrency cluster shared the same AWS IAM role!

---

### Unity Catalog Volumes (Managed vs. External)
A **Volume** is a first-class Unity Catalog object residing within a schema, governing non-tabular files using standard SQL permissions!

$$\text{catalog} \;\; \boldsymbol{.} \;\; \text{schema} \;\; \boldsymbol{.} \;\; \text{volume\_name}$$

| Feature | Managed Volume | External Volume |
| :--- | :--- | :--- |
| **Storage Location** | Stored inside Unity Catalog's default root storage | Points to an external S3 bucket or ADLS path |
| **Lifecycle** | Dropping the volume **deletes the underlying files** | Dropping the volume deletes metadata; **files remain safe in S3** |
| **Best Used For** | Intermediate ML model artifacts, scratch files | Pre-existing hospital imaging archives, vendor feeds |

```sql
-- Creating an External Volume governed by Unity Catalog:
CREATE EXTERNAL VOLUME prod_healthcare.clinical.mri_scans
LOCATION 's3://hospital-imaging-archive-prod/dicom/'
COMMENT 'Governed volume containing patient MRI DICOM files';

-- Granting read access to medical researchers:
GRANT READ VOLUME prod_healthcare.clinical.mri_scans TO GROUP oncology_researchers;
```


In [ ]:
# Simulating Governed Volume Path Operations & SQL Permissions

class GovernedVolumeSimulator:
    """Simulates Unity Catalog Volume file access control"""
    def __init__(self, fq_volume_name, volume_type, location):
        self.name = fq_volume_name
        self.type = volume_type
        self.location = location
        self.files = {}
        self.grants = set()

    def register_file(self, filename, size_mb, phi_contained=True):
        self.files[filename] = {"size_mb": size_mb, "phi": phi_contained}

    def grant(self, principal, privilege):
        self.grants.add((principal, privilege))

    def read_file(self, principal, filename):
        if (principal, "READ VOLUME") not in self.grants and (principal, "ALL PRIVILEGES") not in self.grants:
            return f"[ACCESS DENIED] Principal '{principal}' lacks READ VOLUME on {self.name}"
        if filename not in self.files:
            return f"[404 NOT FOUND] File '{filename}' not found in volume."
        file_info = self.files[filename]
        return f"[FILE STREAM OPENED] {filename} ({file_info['size_mb']} MB) - Audited access for {principal}"

vol = GovernedVolumeSimulator(
    fq_volume_name="prod_healthcare.clinical.patient_imaging",
    volume_type="EXTERNAL",
    location="s3://hospital-imaging-archive-prod/dicom/"
)

vol.register_file("patient_00982_brain_mri.dcm", 245.8, phi_contained=True)
vol.register_file("patient_01104_chest_xray.dcm", 45.2, phi_contained=True)

print("=== Governing Unstructured Lakehouse Volumes ===")
# Test 1: Unauthorized user tries to read MRI scan
print(vol.read_file("junior_data_scientist", "patient_00982_brain_mri.dcm"))

# Test 2: Grant READ VOLUME to authorized physician
vol.grant("dr_smith_oncology", "READ VOLUME")
print(vol.read_file("dr_smith_oncology", "patient_00982_brain_mri.dcm"))


---
## Module 9: Senior Data Engineer Interview Playbook: Governance Q&A

### The 2-Minute Executive Pitch: How to Answer "How do you handle Governance?"

> *"In our modern Lakehouse, we manage governance centrally through Databricks Unity Catalog rather than fragmented, workspace-level permissions.
> 
> 1. **Architecture:** We enforce a 3-level namespace (`catalog.schema.table`) with separate catalogs for environments and domains. Storage access is decoupled from individual users via Storage Credentials and External Locations.
> 2. **Access Control:** We follow the principle of least privilege using RBAC at group levels. To protect HIPAA/PHI, we avoid creating dozens of duplicate views; instead, we implement **Dynamic Column Masks** for sensitive identifiers (SSN, DOB) and **Row Filters** for provider tenant isolation.
> 3. **Auditing & Lineage:** Unity Catalog automatically captures runtime table and column lineage, allowing us to perform instantaneous impact analysis. All query history, logins, and DBU usage are audited via Delta **System Tables** (`system.access.audit`).
> 4. **External Collaboration:** For external actuarial partners, we use **Delta Sharing** for zero-copy, open sharing across clouds without replicating terabytes."*

---

### The 5 Toughest Governance Questions in Senior DE Interviews:

#### Q1: "Why would a user with `SELECT` on a table still get 'Access Denied'?"
* **Answer:** In Unity Catalog, privilege evaluation requires traversal. The user must possess `USE CATALOG` on the parent catalog and `USE SCHEMA` on the parent schema. Without traversal permissions, table-level grants cannot be evaluated.

#### Q2: "How do you satisfy GDPR 'Right to be Forgotten' on Delta Lake if parquet files are immutable?"
* **Answer:** A SQL `DELETE` updates the Delta log and writes new active parquet files, but older parquet files persist in history for Time Travel. To achieve true cryptographic erasure, you must reduce table retention (`delta.deletedFileRetentionDuration = 0 hours`) and run `VACUUM table RETAIN 0 HOURS` to physically shred unreferenced files from object storage.

#### Q3: "What is the performance overhead of Dynamic Column Masking?"
* **Answer:** Negligible. Column masking is resolved at query compile time by the Catalyst optimizer as an inline projection (`CASE WHEN is_member(...) THEN val ELSE mask END`). There is zero physical table rewriting or disk I/O penalty.

#### Q4: "How does Delta Sharing differ from Snowflake Data Sharing?"
* **Answer:** Snowflake Data Sharing is proprietary and requires both the consumer and provider to use Snowflake. Delta Sharing is an **open-source REST protocol**; consumers can query the share using standard Python Pandas, Apache Spark, Power BI, or even Snowflake without vendor lock-in.

#### Q5: "How do you govern non-tabular files (PDFs, ML models, medical scans)?"
* **Answer:** Via Unity Catalog **Volumes** (Managed and External). Volumes replace legacy DBFS mounts and apply the same unified SQL governance, audit trails, and access grants (`READ VOLUME`, `WRITE VOLUME`) to files in cloud storage.


In [ ]:
# Automated Pre-Release Governance Gatekeeper Script
# Run before any new table or pipeline is approved for production release!

def run_governance_audit(table_catalog_name):
    print(f"=== Running Automated Governance Gatekeeper on: {table_catalog_name} ===")
    checks = []
    
    # Check 1: 3-Level Namespace Conformance
    parts = table_catalog_name.split(".")
    is_3_level = len(parts) == 3
    checks.append(("3-Level Namespace Conformance", is_3_level, "Must follow catalog.schema.table"))
    
    # Check 2: Sensitive Column Tagging & Masking
    has_phi_tag = True # Simulated tag check
    checks.append(("PHI Classification Tagged", has_phi_tag, "Must specify HIPAA PHI attribute"))
    
    # Check 3: Table Documentation / Comment
    has_comment = True
    checks.append(("Table Description Populated", has_comment, "COMMENT ON TABLE must be non-empty"))
    
    # Check 4: Row Filter / Tenant Isolation
    has_row_filter = True
    checks.append(("Tenant Row Filter Configured", has_row_filter, "Must prevent cross-tenant data leakage"))
    
    # Summary
    all_passed = all(c[1] for c in checks)
    for name, passed, desc in checks:
        print(f"  [{'PASS' if passed else 'FAIL'}] {name} - {desc}")
        
    print(f"\nGatekeeper Outcome: {'APPROVED FOR PRODUCTION RELEASE' if all_passed else 'REJECTED BY COMPLIANCE'}")
    return all_passed

run_governance_audit("prod_healthcare.silver.claims")
